# Advanced Pipeline Inspection Demo

This notebook demonstrates how to run a multi-stage processing pipeline using in-memory configuration, bypassing the need for external YAML files. We focus on using `get_data()` to inspect data after specific QC milestones.

The data used is the Nelson (unit_397) glider dataset from the BOCD Bio-Carbon Deployment Catalogue.

In [ ]:
# This file is part of the NOC Autonomy Toolbox.
#
# Copyright 2025-2026 National Oceanography Centre and The Contributors
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import os
import sys
import numpy as np
from pathlib import Path
import requests

# --- Friendly Configuration Variables ---
DATA_URL = "https://linkedsystems.uk/erddap/files/Public_OG1_Data_001/Nelson_20240528/Nelson_646_R.nc"
DATA_DIR = Path("../data/OG1")
INPUT_FILE = DATA_DIR / "Nelson_646_R.nc"
TOOLBOX_PATH = "../../src"
OUTPUT_DIRECTORY = "./"

# Set working directory and download data
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True)

if not INPUT_FILE.exists():
    response = requests.get(DATA_URL)
    if response.status_code == 200:
        with open(INPUT_FILE, "wb") as f:
            f.write(response.content)
        print(f"Data downloaded to {INPUT_FILE}")
    else:
        print("Data download failed")

# Import toolbox
sys.path.append(os.path.abspath(TOOLBOX_PATH))
from toolbox.pipeline import Pipeline, _setup_logging

## Virtual Pipeline Setup

Instead of loading from a file, we initialise an empty pipeline and inject the configuration directly. We also initialise logging to the console only, ensuring no log files are generated on the laptop.

In [ ]:
# Define configuration in memory
config = {
    "pipeline": {
        "name": "Virtual Inspection Pipeline",
        "out_directory": OUTPUT_DIRECTORY,
        "visualisation": False
    },
    "steps": [
        {
            "name": "Load OG1",
            "parameters": {"file_path": str(INPUT_FILE.resolve())},
            "diagnostics": False
        }
    ]
}

# Initialise pipeline without a file path
pipe = Pipeline()

# Manually set parameters and logging (no log file specified)
pipe.global_parameters = config["pipeline"]
pipe.logger = _setup_logging(out_dir=OUTPUT_DIRECTORY, log_file=None)
pipe.build_steps(config["steps"])

# Run the initial load step
pipe.run()
print("Initial data loaded.")
data_after_load = pipe.get_data()
data_after_load

## Coordinate Quality Control

We now add and run the first major QC block which focuses on TIME, LATITUDE, and LONGITUDE. After running, we use `get_data()` to inspect the dataset state.

In [ ]:
pipe.add_step(
    step_name="Apply QC",
    parameters={
        "qc_settings": {
            "impossible date qc": {},
            "impossible location qc": {},
            "position on land qc": {},
            "impossible speed qc": {}
        }
    },
    diagnostics=False,
    run_immediately=True
)

# Inspect data after Coordinate QC
data_after_coords = pipe.get_data()
data_after_coords

## CTD Quality Control

Next, we apply complex QC to the CTD variables. This includes range checks and stuck value detection for Pressure (PRES), with flags being propagated to Conductivity (CNDC) and Temperature (TEMP).

In [ ]:
pipe.add_step(
    step_name="Apply QC",
    parameters={
        "qc_settings": {
            "impossible range qc": {
                "variable_ranges": {
                    "PRES": {
                        3: [-5, -2.4],
                        4: [float("-inf"), -5]
                    }
                },
                "also_flag": {"PRES": ["CNDC", "TEMP"]},
                "plot": ["PRES"]
            },
            "stuck value qc": {
                "variables": {"PRES": 2},
                "also_flag": {"PRES": ["CNDC", "TEMP"]},
                "plot": ["PRES"]
            }
        }
    },
    diagnostics=False,
    run_immediately=True
)

# Final data extraction for the next inspection
final_data = pipe.get_data()
final_data

## Automated Flag Validation

One major advantage of `get_data()` is the ability to run custom validation scripts part-way through or at the end of a process. Below, we loop through our core measurement variables to ensure that a corresponding `_QC` variable exists and that every data point has been assigned a valid ARGO flag (0 to 9), ensuring no missing (NaN) flags remain.

In [ ]:
# --- Validation Configuration ---
data = pipe.get_data()

# We only want to check variables that are measurements (have N_MEASUREMENTS dimension)
# Metadata scalars don't need QC columns
measurement_vars = [v for v in data.data_vars if "N_MEASUREMENTS" in data[v].dims and not v.endswith("_QC")]

print(f"Starting Global QC Flag Validation for {len(measurement_vars)} measurement variables...\n")

for var in measurement_vars:
    qc_var = f"{var}_QC"
    
    # 1. Check for existence
    if qc_var not in data.data_vars:
        print(f"[FAILURE] {qc_var} is missing from the dataset.")
        continue
    
    qc_values = data[qc_var].values
    
    # 2. Check for NaNs first
    if np.issubdtype(qc_values.dtype, np.floating) and np.isnan(qc_values).any():
        print(f"[FAILURE] {qc_var} contains bad values!")
    
    # 3. If no NaNs, check the range
    else:
        valid_mask = (qc_values >= 0) & (qc_values <= 9)
        if valid_mask.all():
            print(f"[SUCCESS] {qc_var} is valid (0-9).")
        else:
            invalid_flags = np.unique(qc_values[~valid_mask])
            print(f"[FAILURE] {qc_var} contains invalid flags: {invalid_flags}")

## Summary

By using `get_data()` at each milestone, we verified the addition of QC flags without needing to reload the dataset or write intermediate files to disk. The validation step above confirms that our logic correctly handles flags within the ARGO compliant 0-9 range.